# Mode-Conditioned Drawdown-Drift Short Training Screen v1

This notebook executes the next diagnostic step from `mode_conditioned_drawdown_drift_reward_recipe_screen_v1`: apply the selected drawdown-drift / trend-preservation reward patch to the current top strict retraining candidate, resolve fold-local source checkpoints from the current workspace, and run one primary failed fold short PPO training screen.

This is not a promotion artifact. It is a single-loop diagnostic that checks whether the reward patch improves the primary failed walk-forward fold enough to justify a full three-fold screen.

## Governance

- **Baselines**: canonical baseline `soxl_backtest_policy_reimplementation_v1`; direct comparison `mode_conditioned_rebound_reward_train_v2`.
- **Role**: diagnose.
- **Target window**: primary failed fold `wf_2020` from strict retention handoff; accepted trade-off is lower MaxDD without losing the existing trend participation objective.
- **Changed layer**: reward and model-selection handoff; no runtime execution change.
- **External reference**: PPO reward shaping and behavior-cloning-to-PPO handoff evidence already present in prior SOXL runtime cards.
- **Hypothesis**: the reward patch can reduce controllable drawdown exposure while preserving trend participation; if it fails on `wf_2020`, do not spend compute on a full three-fold run.
- **Acceptance criteria**: source checkpoints resolve locally; diagnostic profile does not overwrite the source profile; `wf_2020` short screen writes a replayable actor/daily CSV and remains non-promotable until official walk-forward gates pass.
- **Historical success case**: `mode_conditioned_rebound_reward_train_v2` is the closest current strict queue candidate.
- **Historical failure case**: old BC/PPO branches improved one dimension but drifted MaxDD, especially on `wf_2022`; this screen avoids continuing that same PPO branch blindly.
- **Not a duplicate**: this run applies the new drawdown-drift/trend-preservation reward patch derived from controllable drawdown rows, and resolves stale checkpoint paths before training.

In [1]:
from __future__ import annotations

import json
import sys
from dataclasses import replace
from pathlib import Path

import pandas as pd

ROOT = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / "src" / "soxl_quant").exists())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.soxl_quant.app.evaluate import slice_frame
from src.soxl_quant.app.short_foldlocal_training_screen import (
    apply_short_training_patch,
    build_short_foldlocal_training_plan,
    render_short_foldlocal_training_plan,
    summarize_short_foldlocal_training_plan,
)
from src.soxl_quant.app.train import run_profile
from src.soxl_quant.app.workers import WALK_FORWARD_WINDOWS
from src.soxl_quant.data import build_dataset, feature_columns_for_packs, init_qlib
from src.soxl_quant.policy.observation import observation_dim_for_profile
from src.soxl_quant.settings import QLIB_DATA_PATH, load_profile

EXPERIMENT = "mode_conditioned_drawdown_drift_short_training_screen_v1"
SOURCE_PROFILE = "mode_conditioned_rebound_reward_train_v2"
TRAINING_PROFILE = "mode_conditioned_drawdown_drift_short_train_v1"
PRIMARY_FOLD = "wf_2020"
CARD_DIR = ROOT / "runtime" / "model_selection" / "card" / EXPERIMENT
SOURCE_MODEL_CARD = ROOT / "runtime" / "model" / "card" / SOURCE_PROFILE
REWARD_PATCH_PATH = ROOT / "runtime" / "reward" / "card" / "mode_conditioned_drawdown_drift_reward_recipe_screen_v1" / "reward_objective_patch.json"
STRICT_HANDOFF_PATH = ROOT / "runtime" / "model_selection" / "card" / "strict_retention_three_fold_handoff_v1" / "summary.json"
STRICT_QUEUE_PATH = ROOT / "runtime" / "model_selection" / "card" / "strict_retraining_candidate_queue_v1" / "summary.json"
FINAL_CONTRACT_PATH = ROOT / "runtime" / "model_selection" / "card" / "final_walkforward_retention_contract_v1" / "summary.json"

CARD_DIR.mkdir(parents=True, exist_ok=True)
reward_patch = json.loads(REWARD_PATCH_PATH.read_text())
strict_handoff = json.loads(STRICT_HANDOFF_PATH.read_text())
strict_queue = json.loads(STRICT_QUEUE_PATH.read_text())
final_contract = json.loads(FINAL_CONTRACT_PATH.read_text())["summary"]
source_profile = load_profile(SOURCE_PROFILE)

print("strict_top_candidate", strict_queue["top_candidate_id"])
print("primary_failed_fold", strict_handoff["primary_failed_fold"])
print("final_retained_count", final_contract["final_retained_count"])
print("reward_patch", reward_patch)


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


strict_top_candidate mode_conditioned_rebound_reward_train_v2
primary_failed_fold wf_2020
final_retained_count 0
reward_patch {'drawdown_drift_penalty': 1.5, 'drawdown_drift_position_threshold': 0.15, 'drawdown_drift_reward_enabled': True, 'drawdown_drift_threshold_pct': 18.0, 'trend_preservation_floor_enabled': True, 'trend_preservation_floor_penalty': 1.0, 'trend_preservation_floor_position': 0.35, 'trend_preservation_lookback': 21, 'trend_preservation_min_buyhold_return_pct': 5.0}


In [2]:
required_folds = tuple(strict_handoff["required_wf_windows"])
plan = build_short_foldlocal_training_plan(
    source_profile=source_profile,
    source_model_card=SOURCE_MODEL_CARD,
    training_profile_name=TRAINING_PROFILE,
    reward_patch=reward_patch,
    required_folds=required_folds,
    max_iters=4,
    stage_iters=2,
)
plan_summary = summarize_short_foldlocal_training_plan(plan)
review = render_short_foldlocal_training_plan(plan, plan_summary)

plan.to_csv(CARD_DIR / "training_plan.csv", index=False)
(CARD_DIR / "training_plan_review.md").write_text(review, encoding="utf-8")
(CARD_DIR / "training_plan_summary.json").write_text(json.dumps(plan_summary, indent=2, sort_keys=True), encoding="utf-8")

plan[["fold", "source_checkpoint_path", "source_checkpoint_exists", "max_iters", "stage_iters"]]


,fold,source_checkpoint_path,source_checkpoint_exists,max_iters,stage_iters
0,wf_2020,/Users/rziiiii/dev/workspace/runtime/model/car...,True,4,2
1,wf_2022,/Users/rziiiii/dev/workspace/runtime/model/car...,True,4,2
2,wf_2023_2024,/Users/rziiiii/dev/workspace/runtime/model/car...,True,4,2


In [3]:
if strict_handoff["primary_failed_fold"] != PRIMARY_FOLD:
    raise RuntimeError(f"expected primary fold {PRIMARY_FOLD}, got {strict_handoff['primary_failed_fold']}")
if not plan_summary["training_allowed"]:
    raise RuntimeError(f"training plan blocked: {plan_summary}")

source_dim = observation_dim_for_profile(source_profile)
renamed_probe = replace(source_profile, name=TRAINING_PROFILE)
renamed_dim = observation_dim_for_profile(renamed_probe)
source_feature_count = len(feature_columns_for_packs(source_profile.feature.packs, source_profile.name))
training_feature_count = len(feature_columns_for_packs(renamed_probe.feature.packs, renamed_probe.name))
feature_schema = {
    "source_obs_dim": int(source_dim),
    "training_obs_dim": int(renamed_dim),
    "source_feature_count": int(source_feature_count),
    "training_feature_count": int(training_feature_count),
    "schema_match": bool(source_dim == renamed_dim and source_feature_count == training_feature_count),
}
(CARD_DIR / "feature_schema_check.json").write_text(json.dumps(feature_schema, indent=2, sort_keys=True), encoding="utf-8")
feature_schema


{'source_obs_dim': 315,
 'training_obs_dim': 315,
 'source_feature_count': 59,
 'training_feature_count': 59,
 'schema_match': True}

In [4]:
init_qlib(QLIB_DATA_PATH)
fold_row = plan[plan["fold"].eq(PRIMARY_FOLD)].iloc[0]
training_profile = apply_short_training_patch(
    source_profile,
    training_profile_name=TRAINING_PROFILE,
    reward_patch=reward_patch,
    checkpoint_path=fold_row["source_checkpoint_path"],
    max_iters=int(fold_row["max_iters"]),
    stage_iters=int(fold_row["stage_iters"]),
)

window_by_name = {name: (train, valid, test) for name, train, valid, test in WALK_FORWARD_WINDOWS}
train_seg, valid_seg, test_seg = window_by_name[PRIMARY_FOLD]

dataset = build_dataset(
    feature_packs=tuple(training_profile.feature.packs),
    profile_name=training_profile.name,
)
full_df = pd.concat([
    dataset.handler.feature_frame.droplevel("instrument").sort_index().loc[train_seg[0]:train_seg[1]],
    dataset.handler.feature_frame.droplevel("instrument").sort_index().loc[valid_seg[0]:valid_seg[1]],
    dataset.handler.feature_frame.droplevel("instrument").sort_index().loc[test_seg[0]:test_seg[1]],
]).sort_index()
price_series = dataset.handler.price_frame.droplevel("instrument")["price"].sort_index()

train_result = run_profile(
    profile=training_profile,
    train_df=slice_frame(full_df, train_seg),
    valid_df=slice_frame(full_df, valid_seg),
    test_df=slice_frame(full_df, test_seg),
    price_series=price_series.loc[test_seg[0]:test_seg[1]],
    smoke=False,
    seed=7,
    artifact_tag=PRIMARY_FOLD,
    retention_gate_evidence={"profile": TRAINING_PROFILE, "summary": {"passing_profiles": []}},
)

primary_metrics = {
    "fold": PRIMARY_FOLD,
    "training_profile": TRAINING_PROFILE,
    "source_profile": SOURCE_PROFILE,
    "source_checkpoint_path": str(fold_row["source_checkpoint_path"]),
    "model_path": str(train_result.get("model_path", "")),
    "summary_path": str(train_result.get("summary_path", "")),
    "review_daily_csv": str(train_result.get("review_daily_csv", "")),
    "return_pct": float(train_result.get("return", 0.0)),
    "sharpe": float(train_result.get("sharpe", 0.0)),
    "max_dd_pct": float(train_result.get("max_dd", 0.0)),
    "promotion_allowed": False,
    "final_model_retained": bool(train_result.get("final_model_retention_decision", {}).get("final_model_retained", False)),
    "final_model_retention_reason": str(train_result.get("final_model_retention_decision", {}).get("reason", "")),
}
(CARD_DIR / "primary_fold_metrics.json").write_text(json.dumps(primary_metrics, indent=2, sort_keys=True), encoding="utf-8")
primary_metrics


[94202:MainThread](2026-06-16 09:39:45,214) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


[94202:MainThread](2026-06-16 09:39:45,730) INFO - qlib.Initialization - [__init__.py:75] - qlib successfully initialized based on client settings.


[94202:MainThread](2026-06-16 09:39:45,730) INFO - qlib.Initialization - [__init__.py:77] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/dev/workspace/data/qlib/soxl')}


[94202:MainThread](2026-06-16 09:39:47,204) INFO - qlib.timer - [log.py:127] - Time cost: 0.001s | Loading data Done


[94202:MainThread](2026-06-16 09:39:47,205) INFO - qlib.timer - [log.py:127] - Time cost: 0.000s | fit & process data Done


...

[94202:MainThread](2026-06-16 09:40:30,967) WARNING - qlib.online operator - [exchange.py:226] - factor.day.bin file not exists or factor contains `nan`. Order using adjusted_price.


[94202:MainThread](2026-06-16 09:40:30,968) WARNING - qlib.online operator - [exchange.py:228] - trade unit 1 is not supported in adjusted_price mode.


backtest loop:   0%|          | 0/252 [00:00<?, ?it/s]

/Users/rziiiii/dev/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/dev/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/dev/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/dev/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/dev/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/dev/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rzi

{'fold': 'wf_2020',
 'training_profile': 'mode_conditioned_drawdown_drift_short_train_v1',
 'source_profile': 'mode_conditioned_rebound_reward_train_v2',
 'source_checkpoint_path': '/Users/rziiiii/dev/workspace/runtime/model/card/mode_conditioned_rebound_reward_train_v2/soxl_mode_conditioned_rebound_reward_train_v2_wf_2020_checkpoint_iter8_actor.pt',
 'model_path': '/Users/rziiiii/dev/workspace/runtime/model/card/mode_conditioned_drawdown_drift_short_train_v1/soxl_mode_conditioned_drawdown_drift_short_train_v1_wf_2020_actor.pt',
 'summary_path': '/Users/rziiiii/dev/workspace/runtime/model/card/mode_conditioned_drawdown_drift_short_train_v1/soxl_mode_conditioned_drawdown_drift_short_train_v1_wf_2020_summary.json',
 'review_daily_csv': '/Users/rziiiii/dev/workspace/runtime/model/card/mode_conditioned_drawdown_drift_short_train_v1/mode_conditioned_drawdown_drift_short_train_v1_wf_2020_soxl_daily.csv',
 'return_pct': 50.08332664590836,
 'sharpe': 1.4029689366479416,
 'max_dd_pct': 23.08630

In [5]:
source_gate_rows = pd.DataFrame(json.loads((ROOT / "runtime" / "reward" / "card" / "mode_conditioned_drawdown_drift_reward_recipe_screen_v1" / "summary.json").read_text())["gate_gaps"])
source_primary = source_gate_rows[source_gate_rows["window"].eq(PRIMARY_FOLD)].iloc[0].to_dict()
short_daily = pd.read_csv(primary_metrics["review_daily_csv"])
short_buyhold_return = (float(short_daily["buyhold_nav"].iloc[-1]) / float(short_daily["buyhold_nav"].iloc[0]) - 1.0) * 100.0
short_relative_return = primary_metrics["return_pct"] - short_buyhold_return
comparison = pd.DataFrame([
    {
        "candidate": SOURCE_PROFILE,
        "kind": "source_replay",
        "fold": PRIMARY_FOLD,
        "return_pct": float(source_primary["return_pct"]),
        "buyhold_return_pct": float(source_primary["buyhold_return_pct"]),
        "max_dd_pct": float(source_primary["max_drawdown_pct"]),
        "relative_return_pct": float(source_primary["relative_return_pct"]),
        "promotion_allowed": False,
    },
    {
        "candidate": TRAINING_PROFILE,
        "kind": "short_training_screen",
        "fold": PRIMARY_FOLD,
        "return_pct": primary_metrics["return_pct"],
        "buyhold_return_pct": float(short_buyhold_return),
        "max_dd_pct": primary_metrics["max_dd_pct"],
        "relative_return_pct": float(short_relative_return),
        "promotion_allowed": False,
    },
])
comparison["max_dd_improvement_pp"] = float(source_primary["max_drawdown_pct"]) - comparison["max_dd_pct"]
comparison["return_delta_vs_source_pp"] = comparison["return_pct"] - float(source_primary["return_pct"])
comparison["relative_delta_vs_source_pp"] = comparison["relative_return_pct"] - float(source_primary["relative_return_pct"])
comparison.to_csv(CARD_DIR / "primary_fold_comparison.csv", index=False)
comparison


,candidate,kind,fold,return_pct,buyhold_return_pct,max_dd_pct,relative_return_pct,promotion_allowed,max_dd_improvement_pp,return_delta_vs_source_pp,relative_delta_vs_source_pp
0,mode_conditioned_rebound_reward_train_v2,source_replay,wf_2020,50.083327,58.093571,23.086304,-8.010244,False,0.000000e+00,0.0,0.0
1,mode_conditioned_drawdown_drift_short_train_v1,short_training_screen,wf_2020,50.083327,58.093571,23.086304,-8.010244,False,1.776357e-14,0.0,0.0


In [6]:
short_row = comparison[comparison["candidate"].eq(TRAINING_PROFILE)].iloc[0]
source_row = comparison[comparison["candidate"].eq(SOURCE_PROFILE)].iloc[0]
primary_gate = bool(short_row["max_dd_pct"] < 20.0 and short_row["return_pct"] >= source_row["return_pct"])
summary = {
    "experiment": EXPERIMENT,
    "decision": "primary_fold_short_training_gate_passed_no_promotion" if primary_gate else "primary_fold_short_training_failed_no_promotion",
    "training_profile": TRAINING_PROFILE,
    "source_profile": SOURCE_PROFILE,
    "primary_fold": PRIMARY_FOLD,
    "training_allowed": bool(plan_summary["training_allowed"]),
    "promotion_allowed": False,
    "actor_retention_allowed": False,
    "primary_gate": primary_gate,
    "source_return_pct": float(source_row["return_pct"]),
    "source_buyhold_return_pct": float(source_row["buyhold_return_pct"]),
    "source_relative_return_pct": float(source_row["relative_return_pct"]),
    "source_max_dd_pct": float(source_row["max_dd_pct"]),
    "short_return_pct": float(short_row["return_pct"]),
    "short_buyhold_return_pct": float(short_row["buyhold_return_pct"]),
    "short_relative_return_pct": float(short_row["relative_return_pct"]),
    "short_max_dd_pct": float(short_row["max_dd_pct"]),
    "max_dd_improvement_pp": float(short_row["max_dd_improvement_pp"]),
    "return_delta_vs_source_pp": float(short_row["return_delta_vs_source_pp"]),
    "relative_delta_vs_source_pp": float(short_row["relative_delta_vs_source_pp"]),
    "final_retained_count": int(final_contract["final_retained_count"]),
    "next_mechanism": "run_full_three_fold_short_training_screen_then_official_wf_gate" if primary_gate else "revise_reward_patch_or_training_checkpoint_before_full_three_fold_run",
    "sidecars": {
        "training_plan": str(CARD_DIR / "training_plan.csv"),
        "training_plan_review": str(CARD_DIR / "training_plan_review.md"),
        "feature_schema_check": str(CARD_DIR / "feature_schema_check.json"),
        "primary_fold_metrics": str(CARD_DIR / "primary_fold_metrics.json"),
        "primary_fold_comparison": str(CARD_DIR / "primary_fold_comparison.csv"),
        "summary": str(CARD_DIR / "summary.json"),
    },
}
(CARD_DIR / "summary.json").write_text(json.dumps(summary, indent=2, sort_keys=True), encoding="utf-8")
review_lines = [
    "# Mode-Conditioned Drawdown-Drift Short Training Screen",
    "",
    "## Decision",
    "",
    f"- decision: `{summary['decision']}`",
    f"- primary_fold: `{PRIMARY_FOLD}`",
    f"- promotion_allowed: `{summary['promotion_allowed']}`",
    f"- final_retained_count: `{summary['final_retained_count']}`",
    "",
    "## Primary Fold Comparison",
    "",
    "| candidate | kind | return | buyhold | relative | MaxDD | DD improvement | return delta |",
    "| --- | --- | ---: | ---: | ---: | ---: | ---: | ---: |",
]
for row in comparison.to_dict("records"):
    review_lines.append(
        f"| {row['candidate']} | {row['kind']} | {row['return_pct']:.2f}% | {row['buyhold_return_pct']:.2f}% | {row['relative_return_pct']:.2f}pp | {row['max_dd_pct']:.2f}% | {row['max_dd_improvement_pp']:.2f}pp | {row['return_delta_vs_source_pp']:.2f}pp |"
    )
review_lines.extend([
    "",
    "## Readout",
    "",
    "- This is one primary-fold short training loop, not a retained model.",
    "- Relative return is recomputed from the review daily CSV to keep the source and short-screen rows on the same buyhold basis.",
    "- Full retention still requires all required walk-forward windows plus the strict retention contract.",
])
(CARD_DIR / "short_training_review.md").write_text("\n".join(review_lines).rstrip() + "\n", encoding="utf-8")
summary


{'experiment': 'mode_conditioned_drawdown_drift_short_training_screen_v1',
 'decision': 'primary_fold_short_training_failed_no_promotion',
 'training_profile': 'mode_conditioned_drawdown_drift_short_train_v1',
 'source_profile': 'mode_conditioned_rebound_reward_train_v2',
 'primary_fold': 'wf_2020',
 'training_allowed': True,
 'promotion_allowed': False,
 'actor_retention_allowed': False,
 'primary_gate': False,
 'source_return_pct': 50.08332664590836,
 'source_buyhold_return_pct': 58.093570616483994,
 'source_relative_return_pct': -8.010243970575637,
 'source_max_dd_pct': 23.086304019655522,
 'short_return_pct': 50.08332664590836,
 'short_buyhold_return_pct': 58.093570616483994,
 'short_relative_return_pct': -8.010243970575637,
 'short_max_dd_pct': 23.086304019655504,
 'max_dd_improvement_pp': 1.7763568394002505e-14,
 'return_delta_vs_source_pp': 0.0,
 'relative_delta_vs_source_pp': 0.0,
 'final_retained_count': 0,
 'next_mechanism': 'revise_reward_patch_or_training_checkpoint_before_

## Conclusion

- Ran: one primary-fold short PPO training screen for `mode_conditioned_drawdown_drift_short_train_v1` on `wf_2020` using the drawdown-drift/trend-preservation reward patch.
- Decision: `primary_fold_short_training_failed_no_promotion`; promotion and actor retention remain disabled.
- Primary comparison: source return `50.08%`, MaxDD `23.09%`; short screen return `50.08%`, MaxDD `23.09%`.
- Effect: MaxDD improvement `0.00pp`, return delta `0.00pp`; MaxDD is still above `20%`.
- Retention: final retained actor count remains `0`.
- Next action: `revise_reward_patch_or_training_checkpoint_before_full_three_fold_run`.
- Artifact links: `training_plan.csv`, `feature_schema_check.json`, `primary_fold_metrics.json`, `primary_fold_comparison.csv`, `summary.json`, and `short_training_review.md`.
